<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 3 章：编写注意力机制

本 Notebook 使用的软件包：

In [129]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.12.0


- 本章介绍注意力机制，它是 LLM 的核心引擎：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="1000px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="600px">

&nbsp;
## 3.1 建模长序列时遇到的问题

- 本节没有代码
- 由于源语言和目标语言的语法结构不同，逐词翻译文本并不可行：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="1000px">

- 在 transformer 模型出现之前，编码器-解码器 RNN 常用于机器翻译任务
- 在这种设置中，编码器处理源语言 token 序列，并用隐藏状态（神经网络中的一种中间层表示）生成上下文向量；解码器再基于这个上下文向量生成目标语言文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="1000px">

&nbsp;
## 3.2 用注意力机制捕获数据依赖关系

- 本节没有代码
- 通过注意力机制，网络中负责生成文本的解码器部分可以有选择地访问全部输入 token；这意味着在生成某个特定输出 token 时，某些输入 token 会比其他 token 更重要：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="1000px">

- Transformer 中的自注意力是一种增强输入表示的技术：序列中的每个位置都可以与同一序列中的其他位置交互，并判断它们的相关性

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="1000px">

&nbsp;
## 3.3 使用自注意力关注输入的不同部分

&nbsp;
### 3.3.1 一个不含可训练权重的简单自注意力机制

- 本节解释一种极简化的自注意力变体，它不包含任何可训练权重
- 这纯粹用于说明概念，并不是 transformer 中实际使用的注意力机制
- 下一节 3.3.2 会扩展这个简单注意力机制，使其能计算所有输入 token 的上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="1000px">

- （请注意，为了减少视觉杂乱，图中的数字只保留到小数点后一位；其他图中也可能包含截断后的数值）

- 按惯例，未归一化的注意力权重称为 **"attention scores"**；而归一化后、总和为 1 的注意力分数称为 **"attention weights"**

- 下面的代码会逐步复现上图

<br>

- **第 1 步：** 计算未归一化的注意力分数 $\omega$
- 假设使用第二个输入 token 作为 query，也就是 $q^{(2)} = x^{(2)}$，通过点积计算未归一化注意力分数

- 假设有下面这个输入句子，并且它已经按第 3 章介绍的方式嵌入为 3 维向量（这里为了便于展示，使用很小的嵌入维度，使它能放在页面上而不换行）：

In [130]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- 本书遵循机器学习和深度学习中的常见约定：训练样本按行表示，特征值按列表示；在上面的张量中，每一行表示一个词，每一列表示一个嵌入维度
- 本节的主要目标是演示如何为第二个输入元素计算上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="1000px">

- 这里以输入序列元素 2，即 $x^{(2)}$，为例计算上下文向量 $z^{(2)}$；本节稍后会把它推广到所有上下文向量
- 第一步是通过计算 query $x^{(2)}$ 和每个输入 token 之间的点积来得到未归一化注意力分数

In [131]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- 补充说明：点积本质上是两个向量逐元素相乘，再把乘积求和的简写：

In [132]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **第 2 步：** 对未归一化注意力分数（"omegas"，$\omega$）进行归一化，使它们的总和为 1
- 下面是一种简单归一化方式，可以让未归一化注意力分数加起来等于 1（这是一种约定，便于解释，也有助于训练稳定性）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="1000px">

In [133]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- 不过在实践中，通常推荐使用 softmax 函数做归一化，因为它能更好地处理极端值，并且在训练期间具有更理想的梯度性质
- 下面是一个朴素的 softmax 实现，它也会把向量归一化为总和为 1：

In [134]:
print(attn_scores_2)
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- 上面的朴素实现面对很大或很小的输入值时，可能因上溢或下溢出现数值不稳定问题
- 因此实践中推荐使用 PyTorch 的 softmax 实现，它已经针对性能进行了高度优化：

In [135]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **第 3 步：** 将嵌入后的输入 token $x^{(i)}$ 与注意力权重相乘并求和，计算上下文向量 $z^{(2)}$：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="1000px">

In [136]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)#zeros 函数创建一个与查询向量相同形状的零向量，用于存储加权求和的结果。
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


&nbsp;
### 3.3.2 计算所有输入 token 的注意力权重

#### 推广到所有输入序列 token：

- 上面我们只为输入 2 计算了注意力权重和上下文向量（如下图高亮行所示）
- 接下来把这个计算推广到所有注意力权重和上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="1000px">

- （请注意，为了减少视觉杂乱，图中的数字截断到小数点后两位；每一行的值应该加起来为 1.0 或 100%；其他图中也可能包含截断后的数值）

- 在自注意力中，流程从计算注意力分数开始，然后将其归一化得到总和为 1 的注意力权重
- 这些注意力权重随后用于对输入进行加权求和，从而生成上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="1000px">

- 将前面的 **第 1 步** 应用于所有成对元素，计算未归一化注意力分数矩阵：

In [137]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 可以通过矩阵乘法更高效地实现上面的计算：

In [138]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 类似前面的 **第 2 步**，对每一行做归一化，使每一行的值加起来为 1：

In [139]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- 快速验证每一行的值确实加起来为 1：

In [140]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- 应用前面的 **第 3 步** 计算所有上下文向量：

In [141]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- 作为 sanity check，之前计算的上下文向量 $z^{(2)} = [0.4419, 0.6515, 0.5683]$ 可以在上面结果的第 2 行找到：

In [142]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


&nbsp;
## 3.4 使用可训练权重实现自注意力

- 下图展示了本节开发的自注意力机制如何融入本书和本章的整体结构

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="1000px">

&nbsp;
### 3.4.1 逐步计算注意力权重

- 本节实现原始 transformer 架构、GPT 模型以及大多数流行 LLM 中使用的自注意力机制
- 这种自注意力机制也称为 "scaled dot-product attention"
- 整体思路与前面类似：我们想为每个输入元素计算上下文向量，但这次会引入可训练权重矩阵

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="1000px">

- 逐步实现自注意力机制时，先介绍三个训练权重矩阵 $W_q$、$W_k$ 和 $W_v$
- 这三个矩阵通过矩阵乘法，把嵌入后的输入 token $x^{(i)}$ 投影成 query、key 和 value 向量

- 输入 $x$ 和 query 向量 $q$ 的嵌入维度可以相同，也可以不同，取决于模型设计和具体实现
- 在 GPT 模型中，输入和输出维度通常相同；但为了更容易跟随计算过程，这里选择较小的输出维度进行演示

In [143]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

- 下面初始化三个权重矩阵；注意这里设置 `requires_grad=False` 是为了让输出更简洁，便于说明
- 如果这些权重矩阵用于模型训练，则应设置 `requires_grad=True`，以便训练期间更新它们

In [144]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)#requires_grad=False 表示这个参数在训练过程中不需要计算梯度，即它是一个固定的参数，不会被更新。
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 接下来计算 query、key 和 value 向量：

In [145]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


- 可以看到，我们成功把 6 个输入 token 从 3 维嵌入空间投影到了 2 维嵌入空间：

In [146]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


- 下一步，也就是 **第 2 步**，通过计算 query 和每个 key 向量之间的点积，得到未归一化注意力分数：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="1000px">

In [147]:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


- 因为有 6 个输入，所以给定 query 向量会对应 6 个注意力分数：

In [148]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="1000px">

- 接下来，在 **第 3 步** 中，使用前面用过的 softmax 函数计算注意力权重（总和为 1 的归一化注意力分数）
- 与前面不同的是，这里会将注意力分数除以嵌入维度平方根 $\sqrt{d_k}$ 进行缩放

In [149]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="1000px">

- 在 **第 4 步** 中，现在计算输入 query 向量 2 的上下文向量：

In [150]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


&nbsp;
### 3.4.2 实现一个紧凑的 SelfAttention 类

- 把所有内容组合起来，可以按如下方式实现自注意力机制：

In [151]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="1000px">

- 可以使用 PyTorch 的 Linear 层简化上面的实现；如果禁用 bias 单元，它等价于矩阵乘法
- 相比手动使用 `nn.Parameter(torch.rand(...))`，`nn.Linear` 的另一个优点是它有更合适的权重初始化方案，有助于模型训练更稳定

In [152]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))#SelfAttention_v2 继承了 PyTorch 的：nn.Module。sa_v2(inputs) 会自动调用 forward()

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


- 注意，`SelfAttention_v1` 和 `SelfAttention_v2` 给出的输出不同，因为它们使用了不同的初始权重矩阵

&nbsp;
## 3.5 使用因果注意力隐藏未来词

- 在因果注意力中，对角线以上的注意力权重会被 mask 掉，确保对任意给定输入，LLM 在用注意力权重计算上下文向量时无法利用未来 token

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="1000px">

&nbsp;
### 3.5.1 应用因果注意力 mask

- 本节把前面的自注意力机制转换成因果自注意力机制
- 因果自注意力确保模型对序列中某个位置的预测只依赖于之前位置的已知输出，而不依赖未来位置
- 在 GPT 这样的 decoder-only LLM 中，这是实现自回归生成的关键

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="1000px">

- 为了演示并实现因果自注意力，继续使用上一节中的注意力分数和权重：

In [153]:
# 为了方便，复用上一节
# SelfAttention_v2 对象中的 query 和 key 权重矩阵
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


- mask 掉未来注意力权重的最简单方法是使用 PyTorch 的 `tril` 函数创建一个 mask：主对角线及其下方元素为 1，主对角线以上元素为 0：

In [154]:
context_length = attn_scores.shape[0]
#ones() 函数创建一个全 1 的张量，形状为 (context_length, context_length)。
#tril() 函数创建一个下三角矩阵，将主对角线以下的元素设为 1，其他元素设为 0。
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- 然后，可以将注意力权重与这个 mask 相乘，把对角线以上的注意力分数置零：

In [155]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


- 不过，如果像上面那样在 softmax 之后应用 mask，会破坏 softmax 创建的概率分布
- Softmax 保证所有输出值加起来为 1；在 softmax 之后 mask，需要再次重新归一化输出，使其总和回到 1，这会让流程变复杂，也可能影响训练稳定性

- 为了确保每一行加起来为 1，可以按如下方式归一化注意力权重：

In [156]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


- 虽然到这里已经完成了因果注意力机制的编码，但可以再看一种更高效的等价方法
- 与其把对角线以上的注意力权重置零再重新归一化，不如在 softmax 之前直接 mask 未归一化注意力分数

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="1000px">

In [157]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


- 如下所示，现在每一行中的注意力权重又能正确地加起来为 1：

In [158]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


&nbsp;
### 3.5.2 使用 dropout mask 额外遮蔽注意力权重

- 此外，我们还会应用 dropout，以降低训练期间的过拟合
- Dropout 可以应用在多个位置：例如计算注意力权重之后，或者注意力权重与 value 向量相乘之后
- 这里会在计算注意力权重之后应用 dropout mask

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="1000px">

- 如果 dropout rate 为 0.5（50%），未被丢弃的值会按因子 1/0.5 = 2 进行相应缩放
- 缩放因子由公式 1 / (1 - `dropout_rate`) 计算得到

In [159]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [160]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


- 注意，根据操作系统不同，得到的 dropout 输出可能看起来不同；可以在 [PyTorch issue tracker 的这里](https://github.com/pytorch/pytorch/issues/121595) 阅读更多关于这个不一致性的内容

&nbsp;
### 3.5.3 实现一个紧凑的因果自注意力类

- 现在可以实现一个可工作的自注意力实现，其中包含因果 mask 和 dropout mask
- 还需要处理由多个输入组成的 batch，使 `CausalAttention` 类支持第 2 章 data loader 产生的批量输出

In [161]:
print(inputs)
#stack() 函数将多个张量按指定维度进行连接。
batch = torch.stack((inputs, inputs), dim=0)
#print(batch)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
torch.Size([2, 6, 3])


[:num_tokens, :num_tokens]切片语法
[0:4, 0:4]
意思是取：
text
第 0 到第 3 行
第 0 到第 3 列
也就是取左上角的 4 x 4 子矩阵。



register_buffer 的作用是：把 mask 注册成模型的一部分，但它不是可训练参数。

也就是说：
self.mask
可以像普通属性一样用，但是它不会被优化器更新。

在ch03\03_understanding-buffers\understanding-buffers.ipynb 中有更详细的解释。


In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        #
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # 如果输入中的 `num_tokens` 超过 `context_length`，下面创建 mask 时会出错
        # 错误会出现在更下面的 mask 创建步骤中。
        # 实践中这不是问题，因为 LLM（第 4-7 章）会确保输入  
        # 在到达这个 forward 方法之前不会超过 `context_length`。 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        #transpose() 函数交换张量的维度。这里我们交换了维度 1 和 2,等价矩阵转置
        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            # 将 mask 中为 True 的位置设为 -torch.inf，这将导致 softmax 输出为 0
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)#0.0 dropout rate for now

context_vecs = ca(batch)

print(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- 注意，dropout 只在训练期间应用，推理期间不会应用

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="1000px">

&nbsp;
## 3.6 将单头注意力扩展为多头注意力

&nbsp;
### 3.6.1 堆叠多个单头注意力层

- 下面总结了前面实现的自注意力（为简洁起见没有显示因果 mask 和 dropout mask）
- 这也称为单头注意力：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="1000px">

- 可以简单地堆叠多个单头注意力模块，形成多头注意力

In [163]:

class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


- 在上面的实现中，嵌入维度是 4，因为我们设置 `d_out=2`，将其作为 key、query、value 向量以及上下文向量的嵌入维度
- 由于有 2 个注意力头，所以输出嵌入维度是 2*2=4

&nbsp;
### 3.6.2 使用权重拆分实现多头注意力

- 虽然上面的实现直观且功能完整（它包装了前面的单头 `CausalAttention` 实现），但也可以编写一个独立的 `MultiHeadAttention` 类来完成同样的事情
- 这里不再拼接多个独立注意力模块，而是通过添加 `num_heads` 维度在内部拆分权重和张量

In [164]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        #out_proj 是一个线性层，用于将多头注意力的输出进行投影，以得到最终的输出向量。它的输入维度是 d_out（即 num_heads * head_dim），输出维度也是 d_out。这一步是可选的，但在实际的 Transformer 模型中通常会使用这个投影层来增加模型的表达能力。
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # 与 `CausalAttention` 一样，如果输入中的 `num_tokens` 超过 `context_length`， 
        # 下面创建 mask 时会出错。 
        # 实践中这不是问题，因为 LLM（第 4-7 章）会确保输入  
        # 在到达这个 forward 方法之前不会超过 `context_length`。

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 通过添加 `num_heads` 维度，隐式地拆分矩阵
        # 展开最后一维：(b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置：(b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 计算 scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- 注意，上面的实现本质上是 `MultiHeadAttentionWrapper` 的更高效重写版本
- 由于随机权重初始化不同，输出看起来会略有不同，但两者都是完整可用的实现，可用于下一章将实现的 GPT 类

---

**关于输出维度的说明**

- 在前面的 `MultiHeadAttentionWrapper` 中，`d_out=2` 表示**每个注意力头**的输出维度；由于会把 `num_heads=2` 个 head 的输出拼接起来，所以最终输出维度是 `d_out * num_heads`，也就是 `2*2=4`
- 在上面的独立 `MultiHeadAttention` 类中，`d_out=2` 表示**所有 head 合并后的最终输出维度**；类内部会把它拆成 `num_heads` 份，因此 `head_dim = d_out // num_heads`
- 所以当 `MultiHeadAttention(d_out=2, num_heads=2)` 时，每个 head 只有 1 维，最终输出维度是 2；这和 `MultiHeadAttentionWrapper(d_out=2, num_heads=2)` 的最终输出维度 4 不一样
- 回头来看，正如读者在 [这个 PR](https://github.com/rasbt/LLMs-from-scratch/pull/859) 中指出的，为了更直观地和 `MultiHeadAttentionWrapper(d_out=2, num_heads=2)` 对齐，使用 `MultiHeadAttention(d_out=4, num_heads=2)` 会更容易理解


- 另外注意，我们还在上面的 `MultiHeadAttention` 类中添加了一个线性投影层（`self.out_proj`）
- 这只是一个不改变维度的线性变换；在 LLM 实现中使用这种投影层是标准惯例，但对本章这个小示例而言并非严格必要

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="1000px">

- 如果你对上面内容的紧凑高效实现感兴趣，也可以考虑 PyTorch 中的 [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 类

- 由于上面的实现乍看起来可能有点复杂，下面看看执行 `attn_scores = queries @ keys.transpose(2, 3)` 时实际发生了什么：

In [165]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- 在这种情况下，PyTorch 的矩阵乘法实现会处理 4 维输入张量：矩阵乘法发生在最后两个维度之间（num_tokens, head_dim），并会对不同 head 重复执行
- 例如，下面是一个更小、更容易观察的例子：

In [166]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


&nbsp;
## 小结与要点

- 参见 [./multihead-attention.ipynb](./multihead-attention.ipynb) 代码 Notebook；它是第 2 章 data loader 加上本章实现的多头注意力类的精简版本，后续章节训练 GPT 模型时会用到
- 练习答案见 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)